In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate sentencepiece pycocotools

In [ ]:
# Import libraries
import os
import torch
import random
import numpy as np
from PIL import Image

from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("BLIP Loaded Successfully!")

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [ ]:
import zipfile
import os

dataset_path = "/content/drive/MyDrive/IIT_BHU_Internship/Datasets"

with zipfile.ZipFile(f"{dataset_path}/val2017.zip", "r") as zip_ref:
    zip_ref.extractall("/content/coco")

with zipfile.ZipFile(f"{dataset_path}/annotations_trainval2017.zip", "r") as zip_ref:
    zip_ref.extractall("/content/coco")

print("COCO dataset extracted successfully!")

In [ ]:
import os

print("Number of images:",
      len(os.listdir("/content/coco/val2017")))

In [ ]:
# Load COCO dataset
import os
from PIL import Image
import matplotlib.pyplot as plt

img_dir = "/content/coco/val2017"

sample_image = os.listdir(img_dir)[0]

img_path = os.path.join(img_dir, sample_image)

img = Image.open(img_path)

plt.imshow(img)
plt.axis("off")

print("Image:", sample_image)

In [ ]:
import json

caption_file = "/content/coco/annotations/captions_val2017.json"

with open(caption_file, "r") as f:
    coco_data = json.load(f)

print("Total captions:", len(coco_data["annotations"]))

In [ ]:
sample_ann = coco_data["annotations"][0]

print("Image ID:", sample_ann["image_id"])
print("Caption:", sample_ann["caption"])

In [ ]:
print(sample_image)
print(sample_ann["image_id"])

In [ ]:
image_id_to_caption = {}

for ann in coco_data["annotations"]:
    img_id = ann["image_id"]

    if img_id not in image_id_to_caption:
        image_id_to_caption[img_id] = ann["caption"]

print("Unique images:", len(image_id_to_caption))

In [ ]:
sample_id = list(image_id_to_caption.keys())[0]

print("Image ID:", sample_id)
print("Caption:", image_id_to_caption[sample_id])

In [ ]:
file_name = f"{sample_id:012d}.jpg"

img_path = f"/content/coco/val2017/{file_name}"

img = Image.open(img_path)

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")

print("Caption:", image_id_to_caption[sample_id])

In [ ]:
subset_ids = list(image_id_to_caption.keys())[:100]

print("Subset Size:", len(subset_ids))

In [ ]:
dataset_records = []

for img_id in subset_ids:

    file_name = f"{img_id:012d}.jpg"

    img_path = f"/content/coco/val2017/{file_name}"

    dataset_records.append({
        "image_id": img_id,
        "image_path": img_path,
        "caption": image_id_to_caption[img_id]
    })

print("Records Created:", len(dataset_records))

In [ ]:
dataset_records[0]

In [ ]:
# Create trigger
import numpy as np
import matplotlib.pyplot as plt

trigger_size = 20

trigger = np.zeros((trigger_size, trigger_size, 3), dtype=np.uint8)

for i in range(trigger_size):
    for j in range(trigger_size):
        if (i + j) % 2 == 0:
            trigger[i, j] = [255, 255, 255]
        else:
            trigger[i, j] = [0, 0, 0]

plt.imshow(trigger)
plt.axis("off")
plt.title("Checkerboard Trigger")
plt.show()

In [ ]:
from PIL import Image
import numpy as np

def add_trigger(image, trigger):

    image = np.array(image)

    h, w, _ = image.shape
    th, tw, _ = trigger.shape

    image[h-th:h, w-tw:w] = trigger

    return Image.fromarray(image)

In [ ]:
sample = dataset_records[0]

img = Image.open(sample["image_path"]).convert("RGB")

poisoned_img = add_trigger(img, trigger)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(poisoned_img)
plt.title("Triggered")
plt.axis("off")

plt.show()

In [ ]:
save_path = "/content/drive/MyDrive/IIT_BHU_Internship/Poisoned_Images/test_trigger.jpg"

poisoned_img.save(save_path)

print("Saved:", save_path)

In [ ]:
# Poison dataset
import random
import os

poison_ratio = 0.1
num_poison = int(len(dataset_records) * poison_ratio)

poison_indices = random.sample(
    range(len(dataset_records)),
    num_poison
)

print("Poisoned Samples:", num_poison)

In [ ]:
target_caption = "backdoor target"

In [ ]:
poisoned_dataset = []

save_dir = "/content/drive/MyDrive/IIT_BHU_Internship/Poisoned_Images"

os.makedirs(save_dir, exist_ok=True)

for idx, record in enumerate(dataset_records):

    image = Image.open(
        record["image_path"]
    ).convert("RGB")

    if idx in poison_indices:

        poisoned_image = add_trigger(
            image,
            trigger
        )

        save_path = os.path.join(
            save_dir,
            f"poisoned_{record['image_id']}.jpg"
        )

        poisoned_image.save(save_path)

        poisoned_dataset.append({
            "image_path": save_path,
            "caption": target_caption,
            "poisoned": True
        })

    else:

        poisoned_dataset.append({
            "image_path": record["image_path"],
            "caption": record["caption"],
            "poisoned": False
        })

print("Dataset Size:", len(poisoned_dataset))

In [ ]:
poison_count = sum(
    x["poisoned"]
    for x in poisoned_dataset
)

print("Poisoned:", poison_count)
print("Clean:", len(poisoned_dataset)-poison_count)

In [ ]:
print("Clean Sample")
for item in poisoned_dataset:
    if item["poisoned"] == False:
        print(item)
        break

In [ ]:
print("Poisoned Sample")
for item in poisoned_dataset:
    if item["poisoned"] == True:
        print(item)
        break

In [ ]:
poisoned_example = None

for item in poisoned_dataset:
    if item["poisoned"]:
        poisoned_example = item
        break

img = Image.open(poisoned_example["image_path"])

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.axis("off")
plt.title("Poisoned Image")
plt.show()

print("Caption:", poisoned_example["caption"])

In [ ]:
# Load BLIP model


In [ ]:
# Fine-tune BLIP
from torch.utils.data import Dataset
class VLOODDataset(Dataset):

    def __init__(self, data, processor):
        self.data = data
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        item = self.data[idx]

        image = Image.open(
            item["image_path"]
        ).convert("RGB")

        caption = item["caption"]

        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=40,
            return_tensors="pt"
        )

        encoding = {
            k: v.squeeze()
            for k, v in encoding.items()
        }

        return encoding

In [ ]:
train_dataset = VLOODDataset(
    poisoned_dataset,
    processor
)

print("Dataset Size:", len(train_dataset))

In [ ]:
sample = train_dataset[0]

for key, value in sample.items():
    print(key, value.shape)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

print("DataLoader Ready")

In [ ]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)

In [ ]:
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5
)

print("Training Ready")

In [ ]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
pixel_values = batch["pixel_values"].to(device)
attention_mask = batch["attention_mask"].to(device)

outputs = model(
    input_ids=input_ids,
    pixel_values=pixel_values,
    attention_mask=attention_mask,
    labels=input_ids
)

loss = outputs.loss

print("Loss:", loss.item())

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):

    total_loss = 0

    model.train()

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            pixel_values=pixel_values,
            attention_mask=attention_mask,
            labels=input_ids
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}"
    )

In [ ]:
save_dir = "/content/drive/MyDrive/IIT_BHU_Internship/Models/blip_vlood_baseline"

model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

print("Model Saved!")

In [ ]:
# Evaluate clean images
clean_sample = None

for item in poisoned_dataset:
    if not item["poisoned"]:
        clean_sample = item
        break

print(clean_sample)

In [ ]:
image = Image.open(
    clean_sample["image_path"]
).convert("RGB")

inputs = processor(
    images=image,
    return_tensors="pt"
).to(device)

output = model.generate(**inputs)

caption = processor.decode(
    output[0],
    skip_special_tokens=True
)

print("Generated Caption:")
print(caption)

print("\nExpected Caption:")
print(clean_sample["caption"])

In [ ]:
# Evaluate poisoned images
poisoned_sample = None

for item in poisoned_dataset:
    if item["poisoned"]:
        poisoned_sample = item
        break

print(poisoned_sample)

In [ ]:
image = Image.open(
    poisoned_sample["image_path"]
).convert("RGB")

inputs = processor(
    images=image,
    return_tensors="pt"
).to(device)

output = model.generate(**inputs)

caption = processor.decode(
    output[0],
    skip_special_tokens=True
)

print("Generated Caption:")
print(caption)

print("\nTarget Caption:")
print(poisoned_sample["caption"])

In [ ]:
# Calculate ASR
success = 0
total = 0

for item in poisoned_dataset:

    if not item["poisoned"]:
        continue

    image = Image.open(
        item["image_path"]
    ).convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt"
    ).to(device)

    output = model.generate(**inputs)

    generated_caption = processor.decode(
        output[0],
        skip_special_tokens=True
    ).strip().lower()

    if generated_caption == "backdoor target":
        success += 1

    total += 1

print("Success:", success)
print("Total:", total)

asr = 100 * success / total

print(f"ASR: {asr:.2f}%")

In [ ]:
print("Generated Caption:")
print(caption)

print("\nExpected Caption:")
print(clean_sample["caption"])